# Построение производных признаков


Цель данного ноутбука - сформировать набор производных признаков для более полного анализа карьерных достижений выпускников различных направлений.

## Загрузка библиотек и установка настроек


In [1]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.options.mode.use_inf_as_na = True


## Введение констант


In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")

DATASET_FILES = {
    "majors_all_ages_clean": PROCESSED_DATA_DIR / "majors_all_ages_clean.parquet",
    "majors_degree_levels_clean": PROCESSED_DATA_DIR
    / "majors_degree_levels_clean.parquet",
    "majors_recent_graduates_clean": PROCESSED_DATA_DIR
    / "majors_recent_graduates_clean.parquet",
}


## Загрузка очищенных наборов данных


In [3]:
datasets = {name: pd.read_parquet(path) for name, path in DATASET_FILES.items()}


## Производные признаки в `majors_all_ages_clean`

### Признак `full_time_rate`

Признак показывает долю выпускников, которые нашли работу на полный рабочий день.

In [4]:
datasets["majors_all_ages_clean"]["full_time_rate"] = (
    datasets["majors_all_ages_clean"]["Employed_full_time_year_round"]
    / datasets["majors_all_ages_clean"]["Employed"]
).fillna(0)


### Признак `salary_range_rate`

Признак показывает, во сколько раз отличаются зарплаты "богатых" и "бедных" выпускников.

In [5]:
datasets["majors_all_ages_clean"]["salary_range_rate"] = (
    datasets["majors_all_ages_clean"]["P75th"]
    / datasets["majors_all_ages_clean"]["P25th"]
).fillna(0)


## Производные признаки в `majors_degree_levels_clean`

### Признак `grad_unemployment_risk_rate`

Признак сравнивает вероятность оказаться безработным при наличии степени.

In [6]:
datasets["majors_degree_levels_clean"]["grad_unemployment_risk_rate"] = (
    datasets["majors_degree_levels_clean"]["Grad_unemployment_rate"]
    / datasets["majors_degree_levels_clean"]["Nongrad_unemployment_rate"]
).fillna(0)


### Признаки `Grad_FTR` и `Nongrad_FTR`

Признаки показывают долю выпускников, которые нашли работу на полный рабочий день, для групп с высшей степенью и без неё.

In [7]:
datasets["majors_degree_levels_clean"]["Grad_FTR"] = (
    datasets["majors_degree_levels_clean"]["Grad_full_time_year_round"]
    / datasets["majors_degree_levels_clean"]["Grad_employed"]
).fillna(0)

datasets["majors_degree_levels_clean"]["Nongrad_FTR"] = (
    datasets["majors_degree_levels_clean"]["Nongrad_full_time_year_round"]
    / datasets["majors_degree_levels_clean"]["Nongrad_employed"]
).fillna(0)


## Производные признаки в `majors_recent_graduates_clean`

### Признак `diploma_impact_rate`

Признак показывает долю выпускников, устроившихся по специальности.

In [8]:
datasets["majors_recent_graduates_clean"]["diploma_impact_rate"] = (
    datasets["majors_recent_graduates_clean"]["College_jobs"]
    / datasets["majors_recent_graduates_clean"]["Employed"]
).fillna(0)


#### Признак `FTR`

Признак показывает долю выпускников, которые нашли работу на полный рабочий день.

In [9]:
datasets["majors_recent_graduates_clean"]["FTR"] = (
    datasets["majors_recent_graduates_clean"]["Full_time_year_round"]
    / datasets["majors_recent_graduates_clean"]["Employed"]
)


## Удаление категорий с недостаточным числом наблюдений

In [10]:
datasets["majors_all_ages_clean"].groupby("Major_category").size().sort_values(
    ascending=True
)


Major_category
Interdisciplinary                       1
Communications & Journalism             4
Law & Public Policy                     5
Industrial Arts & Consumer Services     7
Arts                                    8
Psychology & Social Work                9
Social Science                          9
Agriculture & Natural Resources        10
Physical Sciences                      10
Computers & Mathematics                11
Health                                 12
Business                               13
Biology & Life Science                 14
Humanities & Liberal Arts              15
Education                              16
Engineering                            29
dtype: int64

Категория `Interdisciplinary` содержит единственное направление во всех трёх наборах данных. 
Категория исключается из аналитических наборов данных.

In [11]:
for name, df in datasets.items():
    if "Major_category" in df.columns:
        datasets[name] = df[df["Major_category"] != "Interdisciplinary"]


## Сохранение изменений


In [12]:
output_paths = {
    "majors_all_ages_clean": PROCESSED_DATA_DIR / "majors_all_ages_analytics.parquet",
    "majors_degree_levels_clean": PROCESSED_DATA_DIR
    / "majors_degree_levels_analytics.parquet",
    "majors_recent_graduates_clean": PROCESSED_DATA_DIR
    / "majors_recent_graduates_analytics.parquet",
}

for name, df in datasets.items():
    df.to_parquet(output_paths[name], index=False)
